In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load dataset
df = pd.read_excel("ENB2012_data.xlsx")

# Show first rows
print(df.head())

# Check structure
print(df.info())

# Rename columns (important for clarity)
df.columns = [
    "relative_compactness",
    "surface_area",
    "wall_area",
    "roof_area",
    "overall_height",
    "orientation",
    "glazing_area",
    "glazing_area_distribution",
    "heating_load",
    "cooling_load"
]

print(df.head())

     X1     X2     X3      X4   X5  X6   X7  X8     Y1     Y2
0  0.98  514.5  294.0  110.25  7.0   2  0.0   0  15.55  21.33
1  0.98  514.5  294.0  110.25  7.0   3  0.0   0  15.55  21.33
2  0.98  514.5  294.0  110.25  7.0   4  0.0   0  15.55  21.33
3  0.98  514.5  294.0  110.25  7.0   5  0.0   0  15.55  21.33
4  0.90  563.5  318.5  122.50  7.0   2  0.0   0  20.84  28.28
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X1      768 non-null    float64
 1   X2      768 non-null    float64
 2   X3      768 non-null    float64
 3   X4      768 non-null    float64
 4   X5      768 non-null    float64
 5   X6      768 non-null    int64  
 6   X7      768 non-null    float64
 7   X8      768 non-null    int64  
 8   Y1      768 non-null    float64
 9   Y2      768 non-null    float64
dtypes: float64(8), int64(2)
memory usage: 60.1 KB
None
   relative_compactness

In [3]:
# Basic stats
print(df.describe())

# Correlation (VERY IMPORTANT for story later)
print(df.corr())

       relative_compactness  surface_area   wall_area   roof_area  \
count            768.000000    768.000000  768.000000  768.000000   
mean               0.764167    671.708333  318.500000  176.604167   
std                0.105777     88.086116   43.626481   45.165950   
min                0.620000    514.500000  245.000000  110.250000   
25%                0.682500    606.375000  294.000000  140.875000   
50%                0.750000    673.750000  318.500000  183.750000   
75%                0.830000    741.125000  343.000000  220.500000   
max                0.980000    808.500000  416.500000  220.500000   

       overall_height  orientation  glazing_area  glazing_area_distribution  \
count       768.00000   768.000000    768.000000                  768.00000   
mean          5.25000     3.500000      0.234375                    2.81250   
std           1.75114     1.118763      0.133221                    1.55096   
min           3.50000     2.000000      0.000000              

In [4]:
y = df["heating_load"]
X = df.drop(["heating_load", "cooling_load"], axis=1)

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Features and target
X = df.drop(["heating_load", "cooling_load"], axis=1)
y = df["heating_load"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Regression model
reg_model = LinearRegression()
reg_model.fit(X_train, y_train)

# Predict on test
y_pred = reg_model.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_test, y_pred)
print("MAE:", mae)
print(type(reg_model))

MAE: 4.901130006890691e-14
<class 'sklearn.linear_model._base.LinearRegression'>


In [17]:
feature_importance = pd.Series(reg_model.coef_, index=X.columns)
print(feature_importance.sort_values(ascending=False))

heating_cost                 8.333333e+00
surface_area                 4.213253e-02
relative_compactness         4.626713e-14
glazing_area_distribution   -2.726008e-16
orientation                 -1.138720e-15
overall_height              -7.868543e-15
glazing_area                -9.897078e-15
wall_area                   -4.213253e-02
roof_area                   -8.426506e-02
dtype: float64


In [18]:
# Assume cost per unit energy
cost_per_unit = 0.12  # dollars

# Actual cost
df["heating_cost"] = df["heating_load"] * cost_per_unit

# Predicted cost on test predictions
predicted_cost = y_pred * cost_per_unit

print("Sample predicted cost:", predicted_cost[:5])

Sample predicted cost: [1.9764 1.5804 3.9384 4.9584 2.0028]


In [19]:
def predict_energy(input_data):
    input_df = pd.DataFrame([input_data], columns=X.columns)
    prediction = reg_model.predict(input_df)[0]
    cost = prediction * 0.12
    return prediction, cost

In [20]:
sample_building = X_test.iloc[0].copy()

energy, cost = predict_energy(sample_building)
print("Predicted Energy:", energy)
print("Predicted Cost:", cost)

modified = sample_building.copy()
modified["glazing_area"] = 0.1

new_energy, new_cost = predict_energy(modified)
print("New Energy:", new_energy)
print("New Cost:", new_cost)

savings = cost - new_cost
savings_pct = (savings / cost) * 100

print("Savings:", savings)
print("Savings %:", savings_pct)

Predicted Energy: 16.46999999999996
Predicted Cost: 1.976399999999995
New Energy: 16.469999999999963
New Cost: 1.9763999999999955
Savings: -4.440892098500626e-16
Savings %: -2.2469601793668476e-14


In [10]:
savings = cost - new_cost
savings_pct = (savings / cost) * 100

print("Savings:", savings)
print("Savings %:", savings_pct)

Savings: 0.7274999999999998
Savings %: 32.12127710202418


In [30]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model once
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


def explain_energy_final(energy, cost, savings_pct, glazing_area, overall_height):
    if abs(savings_pct) < 0.1:
        savings_pct = 0.0

    text = (
        f"The building is expected to require {energy:.2f} units of heating energy, "
        f"with an estimated heating cost of ${cost:.2f}. "
    )

    if savings_pct > 0:
        text += (
            f"The scenario suggests a potential cost reduction of {savings_pct:.1f}%, "
            f"which indicates an opportunity to improve energy efficiency. "
        )
    else:
        text += (
            "This scenario does not show a meaningful cost reduction, so the current design may already be close to the baseline. "
        )

    if glazing_area > 0.3:
        text += "A higher glazing area may be increasing heat transfer and raising energy demand. "

    if overall_height > 6:
        text += "Greater building height may also be contributing to higher heating requirements. "

    text += "This analysis can help compare design choices and support cost-aware building decisions."

    return text

In [31]:
final_explanation = explain_energy_final(
    energy,
    cost,
    savings_pct,
    modified["glazing_area"],
    modified["overall_height"]
)

print(final_explanation)

The building is expected to require 16.47 units of heating energy, with an estimated heating cost of $1.98. This scenario does not show a meaningful cost reduction, so the current design may already be close to the baseline. This analysis can help compare design choices and support cost-aware building decisions.


In [32]:
# Predict on FULL dataset
df["predicted_heating"] = reg_model.predict(X)

# Cost calculation
cost_per_unit = 0.12
df["cost"] = df["predicted_heating"] * cost_per_unit

# Baseline (average cost)
baseline_cost = df["cost"].mean()

# Savings
df["savings"] = baseline_cost - df["cost"]
df["savings_pct"] = (df["savings"] / baseline_cost) * 100

# Model error (optional but useful)
df["error"] = abs(df["heating_load"] - df["predicted_heating"])

# Round for clean dashboard
df = df.round({
    "predicted_heating": 2,
    "cost": 2,
    "savings": 2,
    "savings_pct": 2,
    "error": 2
})

# Export CSV
df.to_csv("energy_output.csv", index=False)

print("✅ energy_output.csv created successfully")

✅ energy_output.csv created successfully
